# Backtest Synthetic

## Overview

This notebook demonstrates the public functions in `backtest_synthetic` using compact synthetic trading-rule grids.
- Problem: entry and exit rules are often calibrated on historical simulations, which can overfit in-sample observations and make a strategy too dependent on the past path.
- Approach: under a discrete O-U price process, search for the profit-taking and stop-loss combination that maximizes the rule's Sharpe ratio as the candidate optimal trading rule.
- Synthetic Trading-Rule Experiment: It evaluates selected forecast and half-life settings and identifies the best trading rule for each one.
- Synthetic Trading-Rule Sharpe Ratios: It gives insight on Sharpe ratios for profit-taking and stop-loss combinations.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from src.model_backtesting.backtest_synthetic import (
    synthetic_trading_rule_experiment,
    synthetic_trading_rule_sharpe_ratios,
)

## Synthetic Trading-Rule Experiment

This cell defines a compact synthetic experiment grid.
- `forecasts` contains zero, positive, and negative long-run equilibrium assumptions.
- `half_lives` contains fast, medium, and slow mean-reversion cases.
- `sigma` controls the volatility of the synthetic process shocks.
- `profit_taking_range` and `stop_loss_range` define the trading-rule mesh.
- `max_holding_period` is the vertical barrier used when no horizontal barrier is touched.

In [3]:
forecasts = [0, 5, -5]
half_lives = [5, 25, 100]
sigma = 1.0
num_iterations = 1500
max_holding_period = 100
profit_taking_range = np.linspace(0.5, 7.0, 20)
stop_loss_range = np.linspace(0.5, 7.0, 20)
random_state = 42

This cell runs the synthetic trading-rule experiment.
- Each process setting calls the lower-level Sharpe ratio grid calculation.
- The returned frame appends process metadata to each trading-rule result.


In [4]:
experiment = synthetic_trading_rule_experiment(
    forecasts=forecasts,
    half_lives=half_lives,
    profit_taking_range=profit_taking_range,
    stop_loss_range=stop_loss_range,
    sigma=sigma,
    num_iterations=num_iterations,
    max_holding_period=max_holding_period,
    random_state=random_state,
)

display(experiment.head())
print(f"Number of experiment rows: {len(experiment):,}")

,forecast,half_life,sigma,max_holding_period,profit_taking,stop_loss,mean,std,sharpe_ratio
0,0,5,1.0,100,0.5,0.500000,-0.033213,1.287410,-0.025798
1,0,5,1.0,100,0.5,0.842105,-0.044480,1.393089,-0.031929
2,0,5,1.0,100,0.5,1.184211,0.004553,1.501833,0.003032
3,0,5,1.0,100,0.5,1.526316,0.014462,1.612200,0.008971
4,0,5,1.0,100,0.5,1.868421,0.097195,1.674516,0.058044


Number of experiment rows: 3,600


This cell extracts the best trading rule for each process parameter pair.
- Rows are grouped by the process setting.
- The selected row has the highest Sharpe ratio within that process setting.

In [5]:
best_rules = experiment.loc[
    experiment.groupby(["forecast", "half_life"])["sharpe_ratio"].idxmax()
].reset_index(drop=True)

display(best_rules)

,forecast,half_life,sigma,max_holding_period,profit_taking,stop_loss,mean,std,sharpe_ratio
0,-5,5,1.0,100,0.500000,0.500000,-0.960343,1.092026,-0.879414
1,-5,25,1.0,100,0.500000,0.500000,-0.239435,1.276508,-0.187571
2,-5,100,1.0,100,1.184211,5.973684,-0.139166,3.545611,-0.039250
3,0,5,1.0,100,1.526316,7.000000,2.038870,0.673629,3.026697
4,0,25,1.0,100,2.210526,7.000000,1.032630,3.848383,0.268328
5,0,100,1.0,100,1.526316,7.000000,0.285784,3.773501,0.075734
6,5,5,1.0,100,5.973684,3.578947,6.527782,0.529972,12.317223
7,5,25,1.0,100,4.605263,7.000000,4.361869,2.948463,1.479370
8,5,100,1.0,100,3.921053,7.000000,1.389072,5.190905,0.267597


## Synthetic Trading-Rule Sharpe Ratios

This cell plots Sharpe ratio heat maps for zero long-run equilibrium.
- Zero long-run equilibrium is consistent with market-making intuition, where inventory is held while temporary price deviations revert toward the entry level.
- The practical rule is to wait long enough to realize a small gain while tolerating several times that amount in temporary unrealized losses.
- As the half-life increases, the autoregressive coefficient approaches one, the process becomes closer to a random walk, and the trading-rule surface becomes flatter and less reliable for optimization.

In [ ]:
fig, axes = plt.subplots(1, len(half_lives), figsize=(13, 3.8), sharex=True, sharey=True)

for ax, half_life in zip(axes, half_lives):
    surface = experiment[
        (experiment["forecast"] == 0)
        & (experiment["half_life"] == half_life)
    ].pivot(index="stop_loss", columns="profit_taking", values="sharpe_ratio")
    image = ax.imshow(
        surface,
        origin="lower",
        aspect="auto",
        cmap="Greys_r",
        extent=[
            profit_taking_range.min(),
            profit_taking_range.max(),
            stop_loss_range.min(),
            stop_loss_range.max(),
        ],
    )
    ax.set_title(f"Forecast=0, H-L={half_life}")
    ax.set_xlabel("Profit-taking")

axes[0].set_ylabel("Stop-loss")
fig.colorbar(image, ax=axes, label="Sharpe ratio")
plt.show()


This cell plots Sharpe ratio heat maps for positive long-run equilibrium.
- Positive long-run equilibrium is consistent with a position-taker whose existing position has a favorable expected destination.
- The practical rule is to set a higher profit-taking threshold than in the zero-equilibrium case while allowing a wider stop-loss range for temporary adverse moves.
- As the half-life increases, the autoregressive coefficient approaches one, the process becomes closer to a random walk, and the trading-rule surface becomes flatter and less reliable for optimization.


In [ ]:
fig, axes = plt.subplots(1, len(half_lives), figsize=(13, 3.8), sharex=True, sharey=True)

for ax, half_life in zip(axes, half_lives):
    surface = experiment[
        (experiment["forecast"] == 5)
        & (experiment["half_life"] == half_life)
    ].pivot(index="stop_loss", columns="profit_taking", values="sharpe_ratio")
    image = ax.imshow(
        surface,
        origin="lower",
        aspect="auto",
        cmap="Greys_r",
        extent=[
            profit_taking_range.min(),
            profit_taking_range.max(),
            stop_loss_range.min(),
            stop_loss_range.max(),
        ],
    )
    ax.set_title(f"Forecast=5, H-L={half_life}")
    ax.set_xlabel("Profit-taking")

axes[0].set_ylabel("Stop-loss")
fig.colorbar(image, ax=axes, label="Sharpe ratio")
plt.show()


This cell plots Sharpe ratio heat maps for negative long-run equilibrium.
- Negative long-run equilibrium represents an unfavorable existing position rather than a position a rational trader would newly initiate.
- The practical rule is to avoid large profit-taking thresholds with wide stop-losses, because that combination leaves the position exposed while the process is pulled toward a loss.
- As the half-life increases, the autoregressive coefficient approaches one, the process becomes closer to a random walk, and the trading-rule surface becomes flatter and less reliable for optimization.

In [ ]:
fig, axes = plt.subplots(1, len(half_lives), figsize=(13, 3.8), sharex=True, sharey=True)

for ax, half_life in zip(axes, half_lives):
    surface = experiment[
        (experiment["forecast"] == -5)
        & (experiment["half_life"] == half_life)
    ].pivot(index="stop_loss", columns="profit_taking", values="sharpe_ratio")
    image = ax.imshow(
        surface,
        origin="lower",
        aspect="auto",
        cmap="Greys_r",
        extent=[
            profit_taking_range.min(),
            profit_taking_range.max(),
            stop_loss_range.min(),
            stop_loss_range.max(),
        ],
    )
    ax.set_title(f"Forecast=-5, H-L={half_life}")
    ax.set_xlabel("Profit-taking")

axes[0].set_ylabel("Stop-loss")
fig.colorbar(image, ax=axes, label="Sharpe ratio")
plt.show()
